In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define sizes
Bsizerates = [0, 0.5, 1] 
Elsizes = [50,60,70,80,90,100,110,120,130,140,150,160,170,180,190] 
Ssizes = [100,200,300,400,500,600,700,800,900,1000,1100,1200]
Psizerates = [0,100,200,300,400]#[0, 2]

# Define parameters
electrolyzer_efficiency = 62.7  # % based on 66% LLH H2 coressponds to 50.45 + 2.7 kWh for compressor = 53.15 corresponds to 62.7%
Csize = 8  # Compressor capacity in kg H2/hour
Initial_storage_status = 0

# Load PV hourly data
PV = pd.read_csv("PVhourlyHydrogen500.csv", delimiter=";")
PriceAllYears = pd.read_csv("PriceAllYears.csv", delimiter=";")

# Initialize economic results dataframe
economic_df = pd.DataFrame()

# Loop over all parameter combinations
for Psizerate in Psizerates:
    for Bsizerate in Bsizerates:
        for Esize in Elsizes:
            for Ssize in Ssizes:
                #Setting the pv size to be double the electrolyzer
                #Initial_storage_status = 0.8 * Ssize
                Psize = Psizerate #* Esize
                Bsize = Esize * Bsizerate
                # Initialize lists           
                battery_status = [0]  
                excess_energy = []
                electrolyzer_production = []
                pv_to_electrolyzer = []  
                storage_status = [Initial_storage_status]
                compressor_flow_list = []  
                grid_to_electrolyzer = []  # Track Grid input
    
    
                
                # Create simulation dataframe
                Simh = PV[['Year', 'Month', 'Day', 'Hour', 'Production (Wh)', 'Consumption (kWh H2']].copy()
                # Calculate PV production in kWh
                Simh['PV production (kWh)'] = (Simh['Production (Wh)'] / 1000) * Psize
                Simh['Consumption (kg H2)'] = Simh['Consumption (kWh H2']/33.3
                # Load electricity prices
                Simh = Simh.merge(PriceAllYears, on=['Year', 'Month', 'Day', 'Hour'], how='left')  
                
                # Define grid-to-electrolyzer conditions
                price_threshold = Simh['N4 (EUR/kWh)'].quantile(0.9)  # 20th percentile
                storage_threshold = 0.9 * Ssize 
                for i in range(len(Simh)):
                    prev_battery = battery_status[-1] if i > 0 else 0  
                    prev_storage = storage_status[-1] if i > 0 else Initial_storage_status  
                
                    # Available PV energy
                    pv_production = Simh.loc[i, 'PV production (kWh)']
                
                    # Condition to stop PV to Electrolyzer
                    if (i > 0 and compressor_flow_list[-1] >= Csize) or (prev_storage >= Ssize-Csize):
                        pv_to_el = 0  
                        excess_e = pv_production  
                    else:
                        pv_to_el = min(pv_production, Esize)  
                        excess_e = max(0, pv_production - pv_to_el)  

                    
                    excess_e1 = excess_e
                    # Battery charging
                    charged_battery = min(Bsize, max(0, prev_battery + excess_e1))
                    battery_charge_added = charged_battery - prev_battery
                    excess_e = max(0, excess_e1 - battery_charge_added)  
                
                    # Grid to Electrolyzer logic
                    current_price = Simh.loc[i, 'N4 (EUR/kWh)']
                    #current_storage = storage_status[-1]
                
                   # if current_price < price_threshold and prev_storage < min(storage_threshold,Ssize-Csize):
                   #     h2_needed = max(storage_threshold - prev_storage, Esize/(33.3 / (electrolyzer_efficiency / 100)))
                   #     energy_needed = h2_needed * 33.3 / (electrolyzer_efficiency / 100)  
                   #     grid_input = min(energy_needed-pv_to_el, Esize-pv_to_el)  
                   # else:
                   #     grid_input = 0

                    # Step 1: Calculate how much more energy the electrolyzer can accept
                    remaining_el_capacity = Esize - pv_to_el
                    # Step 2: Use battery energy next
                    battery_used = 0
                    if remaining_el_capacity>0 and prev_storage < (Ssize - Csize):
                        battery_used = min(charged_battery, remaining_el_capacity)
                    new_battery = max(0,charged_battery - battery_used)
                    # Step 3: If still under capacity, use grid (if conditions allow)
                    grid_input = 0
                    if remaining_el_capacity - battery_used > 0 and current_price < price_threshold and prev_storage < min(storage_threshold, Ssize - Csize):
                        grid_input = remaining_el_capacity - battery_used
                    
                    # Total energy for electrolyzer
                    available_energy = pv_to_el + battery_used + grid_input
                    
                    grid_to_electrolyzer.append(grid_input)
                
                    # Total available energy for electrolyzer (PV + battery + grid)
                   # if (i > 0 and compressor_flow_list[-1] < Csize) and (prev_storage < Ssize-Csize):
                   #     available_energy = pv_to_el + prev_battery + grid_input
                   # else:
                   #     available_energy = 0
                    electrolyzer_energy = min(available_energy, Esize)
                
                    # Update battery after electrolyzer consumption
                   # new_battery -= max(0, Esize - pv_to_el - grid_input)
                   # new_battery = max(0, new_battery)  
                
                    # Hydrogen production calculation
                    h2_produced = electrolyzer_energy * (electrolyzer_efficiency / 100) * (1 / 33.3)
                    compressor_flow = min(h2_produced, Csize)  
                    compressor_flow_list.append(compressor_flow)  

                    #Consumption definition including the H2 sell
                    if prev_storage > 0.8 * Ssize and Simh.loc[i, 'Consumption (kg H2)'] == 0:
                        Simh.loc[i, 'Consumption (kg H2)'] = Esize * (electrolyzer_efficiency / 100) * (1 / 33.3)
                        Simh.loc[i, 'SellH2 (kg H2)'] = Esize * (electrolyzer_efficiency / 100) * (1 / 33.3)
                
                    
                
                    # Update H2 storage
                    new_storage = min(Ssize, max(0, prev_storage + compressor_flow - Simh.loc[i, 'Consumption (kg H2)']))
                    storage_status.append(new_storage)  
                
                    # Store results
                    battery_status.append(new_battery)
                    excess_energy.append(excess_e)
                    electrolyzer_production.append(electrolyzer_energy)
                    pv_to_electrolyzer.append(pv_to_el)
                
                # Add computed values to DataFrame
                Simh['Battery status (kWh)'] = battery_status[1:]
                Simh['Excess e (kWh)'] = excess_energy
                Simh['Electrolyzer production (kWh)'] = electrolyzer_production
                Simh['PV to Electrolyzer (kWh)'] = pv_to_electrolyzer
                Simh['Grid to Electrolyzer (kWh)'] = grid_to_electrolyzer  # Add Grid input
                Simh['Hydrogen produced (kg)'] = Simh['Electrolyzer production (kWh)'] * electrolyzer_efficiency / 100 * (1 / 33.3)
                Simh['Compressor flow (kg H2)'] = compressor_flow_list  
                Simh['H2 storage (kg H2)'] = storage_status[1:]  
                
                # Initialize H2def list
                H2def = [0]  
                
                for i in range(1, len(Simh)):  
                    prev_storage = Simh.loc[i - 1, 'H2 storage (kg H2)']
                    current_storage = Simh.loc[i, 'H2 storage (kg H2)']
                    consumption = Simh.loc[i, 'Consumption (kg H2)']
                    compressor_flow = Simh.loc[i, 'Compressor flow (kg H2)']
                
                    h2_consumed = max(0, prev_storage - current_storage + compressor_flow)
                    deficit = max(0, consumption - h2_consumed)
                
                    H2def.append(deficit)
                
                Simh['H2def (kg H2)'] = H2def
                
                # Save updated dataframe
                Simh.to_csv("Simh.csv", index=False, sep=";")
                
            
                
                
                # Preparing the economical results              
                # Economic Parameters
                Discount_Rate = 0.05  
                Lifetime = 30  # Project lifetime in years
                
                # CAPEX and OPEX Definitions
                H2_Storage_Capex = 1100 * Ssize  # EUR/kg H2
                H2_Storage_Lifetime = 30  # years
                Annual_H2_Storage_Opex = 1 * H2_Storage_Capex / 100  # % of CAPEX per year
                
                Battery_Capex = 800 * Bsize  # EUR/kWh
                Battery_Lifetime = 30  # years (doubled from original 15 * 2)
                Annual_Battery_Opex = 2 * Battery_Capex / 100 # % of CAPEX per year
                
                Compressor_Capex = 240000  # EUR per unit
                Compressor_Lifetime = 30  # years
                Annual_Compressor_Opex = 0.5 * Compressor_Capex / 100  # % of CAPEX per year
                
                PV_Capex = 900 * Psize  # EUR/kWp
                PV_Lifetime = 30  # years
                Annual_PV_Opex = 1.5 * PV_Capex / 100 # % of CAPEX per year
                
                Electrolyzer_Lifetime = 30  # years
                # Electrolyzer CAPEX Calculation (based on size)
                Electrolyzer_Capex_unit = 1.1195 * (585.85 + (Esize ** 0.622) * (9485.2 / Esize))
                Electrolyzer_CAPEX = Esize * Electrolyzer_Capex_unit
                Annual_Electrolyzer_Opex = 2 * Electrolyzer_CAPEX / 100 # % of CAPEX per year
                Electrolyzer_Replacement_Cost = 0.35 * Electrolyzer_CAPEX  # 35% of initial CAPEX per replacement
                Electrolyzer_Water_Consumption = 10 #liter per kg H2
                Water_Cost = 0.00053  #EUR/liter 
                
                #N3T electricity fee
                Monthly_N3T_Fixed_Fee = 359 # EUR2024/month calculated from 4100 kr per month
                Monthly_N3T_Power_Fee = 3 * Esize # Given in EUR2024/month calculated from 34 kr/kw
                Monthly_N3T_HighLoad_Fee = 7.5 * Esize # Given in EUR2024/month calculated from 86 kr/kw
                Annual_N3T_Electricity_Fixed_Cost = 12 * (Monthly_N3T_Fixed_Fee + Monthly_N3T_Power_Fee + Monthly_N3T_HighLoad_Fee)
                
                #N4 electricity fee
                Monthly_N4_Fixed_Fee = 42 # EUR2024/month calculated from 480 kr per month
                Monthly_N4_Power_Fee = 4.1 * Esize # Given in EUR2024/month calculated from 47 kr/kw
                Monthly_N4_HighLoad_Fee = 0 * Esize # Given in EUR2024/month calculated from 0 kr/kw
                Annual_N4_Electricity_Fixed_Cost = 12 * (Monthly_N4_Fixed_Fee + Monthly_N4_Power_Fee + Monthly_N4_HighLoad_Fee)
                
                
                
                # Total CAPEX
                Total_CAPEX = PV_Capex + Battery_Capex + Electrolyzer_CAPEX + H2_Storage_Capex + Compressor_Capex
                
                
                
                # OPEXfix is the fixed annual operating expenditure
                Total_annual_OPEX_fix = Annual_Electrolyzer_Opex + Annual_PV_Opex + Annual_Compressor_Opex + Annual_Battery_Opex + Annual_H2_Storage_Opex + Annual_N4_Electricity_Fixed_Cost
                
                # Discounting OPEX over the lifetime
                Total_Discounted_OPEX_fix = Total_annual_OPEX_fix * sum([1 / (1 + Discount_Rate) ** t for t in range(1, Lifetime + 1)])
                
                
                # OPEXvar is the variable annual operating expenditure (grid electricity and water for the electrolyser)
                Total_Discounted_Replacement_Cost = Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 10 + Electrolyzer_Replacement_Cost / (1 + Discount_Rate) ** 20
                
                # Calculate total electricity cost from grid usage
                Simh['Electricity Cost (EUR)'] = Simh['Grid to Electrolyzer (kWh)'] * Simh['N4 (EUR/kWh)']
                Total_Electricity_Cost = Simh['Electricity Cost (EUR)'].sum()
                # Calculate total income from selling excess electricity
                Simh['Electricity Income (EUR)'] = Simh['Excess e (kWh)'] * Simh['Income (EUR/kWh)']
                Total_Electricity_Income = Simh['Electricity Income (EUR)'].sum()
                # Divide by 14 for average annual values
                Annual_Electricity_Cost = Total_Electricity_Cost / 14
                Annual_Electricity_Income = Total_Electricity_Income / 14
                Total_Discounted_Electricity_Cost = Annual_Electricity_Cost * sum([1 / (1 + Discount_Rate) ** t for t in range(1, Lifetime + 1)])
                Total_Discounted_Electricity_Income = Annual_Electricity_Income * sum([1 / (1 + Discount_Rate) ** t for t in range(1, Lifetime + 1)])
                
                # Water cost 
                Annual_H2_Produced = Simh['Hydrogen produced (kg)'].sum()/14  # Total H2 produced over the simulation
                Annual_Water_Cost = Electrolyzer_Water_Consumption * Water_Cost * Annual_H2_Produced
                Total_Discounted_Water_Cost = Annual_Water_Cost * sum([1 / (1 + Discount_Rate) ** t for t in range(1, Lifetime + 1)])
                
                
                
                Total_CAPEX
                Total_Discounted_OPEX_fix
                Total_Discounted_Replacement_Cost
                Total_Discounted_Electricity_Cost
                Total_Discounted_Water_Cost
                Total_Discounted_Electricity_Income
                
                # Levelized Cost of Hydrogen (LCOH) 
                Total_H2_Produced = Lifetime * Annual_H2_Produced
                Net_Present_Cost= Total_CAPEX + Total_Discounted_OPEX_fix + Total_Discounted_Replacement_Cost + Total_Discounted_Electricity_Cost + Total_Discounted_Water_Cost -Total_Discounted_Electricity_Income
                if Total_H2_Produced > 0:
                    LCOH = (Net_Present_Cost) / Total_H2_Produced
                else:
                    LCOH = np.nan  # Avoid division by zero
                
                
                economic_results = {
                    'Discount Rate': [Discount_Rate],
                    'Battery Size (kWh)': [Bsize],
                    'Electrolyzer Size (kWp)': [Esize],
                    'Storage Size (kg H2)': [Ssize],
                    'PV Size (kWp)': [Psize],
                    'LCOH (EUR2024/kg H2)': [LCOH],
                    'Net Present Cost (EUR2024)': [Net_Present_Cost], 
                    'Total CAPEX (EUR2024)': [Total_CAPEX],
                    'PV Capex':PV_Capex,
                    'Battery Capex':Battery_Capex,
                    'Electrolyzer Capex':Electrolyzer_CAPEX,
                    'H2 Storage Capex':H2_Storage_Capex,
                    'Compressor Capex':Compressor_Capex,
                    'Total Discounted Fixed OPEX (EUR2024)': [Total_Discounted_OPEX_fix],
                    'Annual Electrolyzer Opex':Annual_Electrolyzer_Opex,
                    'Annual PV Opex':Annual_PV_Opex,
                    'Annual Compressor Opex':Annual_Compressor_Opex,
                    'Annual Battery Opex':Annual_Battery_Opex,
                    'Annual H2 Storage Opex':Annual_H2_Storage_Opex,
                    'Annual N4 Electricity Fixed Cost':Annual_N4_Electricity_Fixed_Cost,
                    'Total Discounted Replacement Cost (EUR2024)': [Total_Discounted_Replacement_Cost],
                    'Total Discounted Electricity Cost (EUR2024)': [Total_Discounted_Electricity_Cost],
                    'Total Discounted Water Cost (EUR2024)': [Total_Discounted_Water_Cost],
                    'Total Discounted Electricity Income (EUR2024)': [Total_Discounted_Electricity_Income],
                    'Total H2 produced (kg)': [Total_H2_Produced],
                    'Total H2 deficit (kg)': [30*sum(Simh['H2def (kg H2)'])/14],
                    'Electrolyzer usage (%)': [100*(sum(Simh['Electrolyzer production (kWh)'])/14)/(8760*Esize)],
                    'Electrolyzer production (kWh)': [30*sum(Simh['Electrolyzer production (kWh)'])/14], 
                    'PV to Electrolyzer (kWh)': [30*sum(Simh['PV to Electrolyzer (kWh)'])/14], 
                    'Grid to Electrolyzer (kWh)':[30*sum(Simh['Grid to Electrolyzer (kWh)'])/14],
                    'Consumption (kg)': [30*sum(Simh['Consumption (kg H2)'])/14],
                    'PV production (kWh)':[30*sum(Simh['PV production (kWh)'])/14],
                    'Excess electricity':[30*sum(Simh['Excess e (kWh)'])/14],
                    'SellH2 (kg H2)': [30*sum(Simh['SellH2 (kg H2)'].fillna(0))/14]
                }
                economic_df = pd.concat([economic_df, pd.DataFrame([economic_results])], ignore_index=True)

#Save to file
economic_df.to_csv("economic_results_SellH2.csv", index=False, sep=";")


In [4]:
economic_df

,Discount Rate,Battery Size (kWh),Electrolyzer Size (kWp),Storage Size (kg H2),PV Size (kWp),LCOH (EUR2024/kg H2),Net Present Cost (EUR2024),Total CAPEX (EUR2024),PV Capex,Battery Capex,...,Total H2 produced (kg),Total H2 deficit (kg),Electrolyzer usage (%),Electrolyzer production (kWh),PV to Electrolyzer (kWh),Grid to Electrolyzer (kWh),Consumption (kg),PV production (kWh),Excess electricity,SellH2 (kg H2)
0,[0.05],[0],[160],[300],[0],[4.403130648966205],[3139069.1579421842],[924416.0069072541],0,0,...,[712917.55984556],[5.0978269225002546e-11],[90.04729288975864],[37863085.71428572],[0.0],[37863085.71428572],[712397.3398584621],[0.0],[0.0],[609492.4169884172]
1,[0.05],[0],[160],[300],[160],[4.352504397290819],[3122652.3556042695],[1068416.006907254],144000,0,...,[717438.0702631661],[6.613519256047442e-11],[90.61827016470971],[38103170.23885714],[4495773.953142857],[33607396.28571428],[716922.7143758367],[4495773.953142857],[0.0],[614017.7915057915]
2,[0.05],[0],[160],[300],[320],[4.3436491328119065],[3134346.338776719],[1212416.006907254],288000,0,...,[721592.8918152896],[7.097893701362473e-11],[91.1430579582518],[38323833.01028572],[7992204.075428571],[30331628.93485714],[721073.6641827865],[8991547.906285714],[999343.8308571428],[618168.7413127413]
3,[0.05],[0],[160],[300],[480],[4.408626706686823],[3192254.969102698],[1356416.006907254],432000,0,...,[724092.8256095754],[7.025427858512297e-11],[91.45881995270712],[38456604.613714285],[9675741.558857143],[28780863.054857142],[723578.4363835587],[13487321.859428572],[3811580.300571428],[620673.5135135136]
4,[0.05],[0],[160],[300],[640],[4.4991109435979375],[3264888.4613163774],[1500416.006907254],576000,0,...,[725674.1392345945],[6.767158262326639e-11],[91.65855273972603],[38540588.256],[10720030.683428572],[27820557.57257143],[725153.6024067246],[17983095.81257143],[7263065.129142857],[622248.6795366796]
